# 3D toric code — energy kink: $E/N$ vs field

The field couples linearly, $H(h)=H_0-h\,M$, so a **first-order** transition shows a **corner (kink) in $E(h)$** — its slope $dE/dh=-\langle M\rangle$ jumps — while a **second-order** one stays smooth. Here we just plot $E/N$ and look. The kink sharpens with $L$, so compare sizes. (Derivative/FSS analysis + references live in `notes/distinguishing_transition_order.md`.)

In [ ]:
# ====================== 1 · CONFIG — the one cell to edit ======================
import json, glob, os
import numpy as np
import matplotlib.pyplot as plt

ROOT = "/Users/sanzhar123/Desktop/Approximate-Symmetries-TC-main/results"

LINES = {
    "1st  (hx @ hz=0)":   dict(order="first",  field="hx", energy=f"{ROOT}/energy_hz0.0"),
    "2nd  (hz @ hx=0.6)": dict(order="second", field="hz", energy=f"{ROOT}/energy_hx0.6"),
}

# ---- which L to include per line (None = all found; drop non-converged sizes here) ----
LS = {
    "1st  (hx @ hz=0)":   [5, 6],     # L=7 not converged on this line -> excluded
    "2nd  (hz @ hx=0.6)": None,       # None = use every L that was pulled
}

# ---- axis zoom per line: (min, max) or None = autoscale ----
XLIM = {"1st  (hx @ hz=0)": None, "2nd  (hz @ hx=0.6)": None}
YLIM = {"1st  (hx @ hz=0)": None, "2nd  (hz @ hx=0.6)": None}

N_EDGES = lambda L: 3*L**2*(L-1)   # OBC cubic 3D-TC qubit count (3L^3 - 3L^2)
CMAP    = plt.cm.viridis
print("lines:", list(LINES))

## 2 · Load the energy curves

In [ ]:
# ====================== 2 · DATA ======================
def load_energy(directory):
    """Compact energy curves from nersc/extract_energy.sh, keyed by L. {} if absent."""
    recs = {}
    for jp in sorted(glob.glob(os.path.join(directory, "*.json"))):
        d = json.load(open(jp))
        recs[int(d["L"])] = dict(h=np.array(d["field"], float), E=np.array(d["E"], float))
    return recs

def sel_Ls(name, available):
    """Sorted L for `name`, filtered by the LS knob (None = all available)."""
    want = LS.get(name)
    return [L for L in sorted(available) if want is None or L in want]

DATA = {name: dict(spec, curves=load_energy(spec["energy"])) for name, spec in LINES.items()}

for name, D in DATA.items():
    found = sorted(D["curves"]); used = sel_Ls(name, D["curves"])
    tag = found or "MISSING -> run nersc/extract_energy.sh + pull"
    print(f"[{name}]  sweep={D['field']}  found L={tag}  ->  using {used}")

## 3 · $E/N$ vs field

One panel per line. Edit **`LS`** to drop sizes, **`XLIM`/`YLIM`** to zoom.

In [ ]:
# ====================== 3 · E/N PLOT ======================
fig, ax = plt.subplots(1, len(DATA), figsize=(7*len(DATA), 5), squeeze=False)
for j, (name, D) in enumerate(DATA.items()):
    a = ax[0, j]
    Ls = sel_Ls(name, D["curves"])
    colors = CMAP(np.linspace(0, 0.85, max(1, len(Ls))))
    for L, c in zip(Ls, colors):
        r = D["curves"][L]; o = np.argsort(r["h"])
        a.plot(r["h"][o], r["E"][o]/N_EDGES(L), "o-", ms=5, lw=1.3, color=c, label=f"L={L}")
    a.set(xlabel=f"${D['field']}$", ylabel="$E/N$",
          title=f"{name}   [{D['order']}]")
    if XLIM.get(name): a.set_xlim(*XLIM[name])
    if YLIM.get(name): a.set_ylim(*YLIM[name])
    if a.has_data(): a.legend(fontsize=9)
    else: a.text(0.5, 0.5, "no energy pulled", ha="center", transform=a.transAxes)
plt.tight_layout(); plt.show()